In [5]:
import folium
import numpy as np
from scipy.spatial import distance

In [6]:
depots = np.array([[10.77715, 106.65675], [10.76553, 106.70771]])
delivery_points = np.random.uniform(low=[10.75, 106.63], high=[10.82, 106.72], size=(20,2))
distance_to_depots = distance.cdist(delivery_points, depots)
assigns = np.argmin(distance_to_depots, axis=1)

optimized_routes = []
total_optimized_routes = 0

In [9]:
for i, depot in enumerate(depots):
  cluster_points = delivery_points[assigns == i].tolist()
  route = [depot.tolist()]
  current_location = depot.tolist()

  while cluster_points:
    dist = distance.cdist([current_location], cluster_points)[0]
    nearest_point_index = np.argmin(dist)
    nearest_point = cluster_points.pop(nearest_point_index)
    total_optimized_routes += dist[nearest_point_index]
    route.append(nearest_point)
    current_location = nearest_point
  total_optimized_routes += distance.euclidean(current_location, depot.tolist())
  route.append(depot.tolist())
  optimized_routes.append(route)

In [18]:
m = folium.Map(location=[10.785, 106.675], zoom_start=13)
colors = ['blue', 'green']
for i,route in enumerate(optimized_routes):
  folium.PolyLine(
      locations=route, color=colors[i],
      tooltip=f"Tuyến xe của Kho {i+1}"
  ).add_to(m)

for i, depot in enumerate(depots):
  folium.Marker(
      location=depot, tooltip=f"Kho hàng {i+1}",
      icon=folium.Icon(color=colors[i], icon='home')
  ).add_to(m)

for point, assign in zip(delivery_points, assigns):
    folium.CircleMarker(
        location=point, radius=3, color='black'
    ).add_to(m)

In [19]:
m